# ✅ Análisis Semántico — `Expresión = Expresión + Expresión` (Google Colab)

Objetivo: mostrar **solo semántica** (tipado) con un parser mínimo a AST.

In [ ]:

# Dependencias
!pip -q install graphviz ipywidgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")
import ipywidgets as widgets
display(HTML("<div style='padding:8px;border-left:4px solid #4CAF50'>✅ Dependencias listas.</div>"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.9 MB/s eta 0:00:00


In [ ]:

# Tokenizador
import re, pandas as pd
token_specs = [
    ("EQ",   r"="), ("PLUS", r"\+"), ("MUL",  r"\*"),
    ("LP",   r"\("), ("RP",   r"\)"),
    ("NUM",  r"\d+(?:\.\d+)?"), ("ID",   r"[a-zA-Z_]\w*"),
    ("SKIP", r"[ \t\n]+"), ("MISMATCH", r"."),
]
tok_master = "|".join(f"(?P<{name}>{pat})" for name, pat in token_specs)
def lex(source: str):
    tokens, ids, seen = [], [], set()
    pos = 0
    for m in re.finditer(tok_master, source):
        kind, text = m.lastgroup, m.group()
        start, end = m.start(), m.end()
        if kind == "SKIP": continue
        if kind == "MISMATCH": raise SyntaxError(f"Token inválido '{text}' en posición {start}")
        if kind == "ID" and text not in seen: ids.append(text); seen.add(text)
        tokens.append((kind, text, start, end)); pos = end
    tokens.append(("EOF", "$", pos, pos)); return tokens, ids


In [ ]:

# Parser mínimo a AST
from dataclasses import dataclass, field
from typing import List, Optional
import graphviz

@dataclass
class Node:
    kind: str
    value: Optional[str] = None
    children: List["Node"] = field(default_factory=list)
    type: Optional[str] = None
    def add(self, *chs):
        for c in chs:
            if c is not None: self.children.append(c)
        return self

class Parser:
    # S -> ID '=' E ; E -> T (+ T)* ; T -> F (* F)* ; F -> '(' E ')' | ID | NUM
    def __init__(self, tokens): self.tokens, self.pos = tokens, 0
    def peek(self): return self.tokens[self.pos]
    def match(self, k):
        if self.peek()[0] == k:
            t = self.tokens[self.pos]; self.pos += 1; return t
        raise SyntaxError(f"Se esperaba {k} pero llegó {self.peek()[0]} en pos {self.peek()[2]}")
    def accept(self, k):
        if self.peek()[0] == k:
            t = self.tokens[self.pos]; self.pos += 1; return t
        return None
    def parse(self):
        left = self.match("ID")[1]; self.match("EQ"); expr = self.parse_E(); self.match("EOF")
        return Node("Assign", value=left).add(expr)
    def parse_E(self):
        n = self.parse_T()
        while self.accept("PLUS"): n = Node("Add").add(n, self.parse_T())
        return n
    def parse_T(self):
        n = self.parse_F()
        while self.accept("MUL"): n = Node("Mul").add(n, self.parse_F())
        return n
    def parse_F(self):
        if self.accept("LP"):
            e = self.parse_E(); self.match("RP"); return e
        tok = self.peek()
        if tok[0] == "ID": self.match("ID"); return Node("Id", value=tok[1])
        if tok[0] == "NUM": self.match("NUM"); return Node("Num", value=tok[1])
        raise SyntaxError(f"Factor inválido en posición {tok[2]}")


In [ ]:

# Semántica
T_INT, T_FLOAT = "int", "float"
def num_type_of_literal(txt): return T_FLOAT if "." in txt else T_INT
def promote(t1, t2): return T_FLOAT if (t1==T_FLOAT or t2==T_FLOAT) else T_INT
def compatible_assign(lhs, rhs): return lhs==rhs or (lhs==T_FLOAT and rhs==T_INT)

def annotate_types(node: Node, symtab: dict, errors: list):
    k = node.kind
    if k == "Num":
        node.type = num_type_of_literal(node.value); return node.type
    if k == "Id":
        t = symtab.get(node.value)
        if not t: errors.append(f"Identificador no declarado: '{node.value}'"); node.type=None
        else: node.type=t
        return node.type
    if k in ("Add","Mul"):
        tL = annotate_types(node.children[0], symtab, errors)
        tR = annotate_types(node.children[1], symtab, errors)
        if tL and tR:
            if tL not in (T_INT,T_FLOAT): errors.append(f"Operando izq no numérico en {k}: {tL}")
            if tR not in (T_INT,T_FLOAT): errors.append(f"Operando der no numérico en {k}: {tR}")
            node.type = promote(tL, tR)
        else:
            node.type = tL or tR
        return node.type
    if k == "Assign":
        expr_t = annotate_types(node.children[0], symtab, errors)
        lhs_t = symtab.get(node.value)
        if not lhs_t:
            errors.append(f"Identificador no declarado en asignación: '{node.value}'")
        elif expr_t and not compatible_assign(lhs_t, expr_t):
            errors.append(f"Asigna inválida: no se puede asignar {expr_t} a {lhs_t} en '{node.value}'")
        node.type = lhs_t; return node.type
    # Propagación por defecto
    acc=None
    for ch in node.children:
        t = annotate_types(ch, symtab, errors); acc = acc or t
    node.type = acc; return node.type


In [ ]:

# Visualización AST
def render_ast(node: Node):
    dot = graphviz.Digraph(graph_attr={"rankdir":"TB"})
    def build(n: Node):
        nid = str(id(n))
        label = n.kind if n.value is None else f"{n.kind}\\n‘{n.value}’"
        if n.type: label += f"\\n<{n.type}>"
        dot.node(nid, label)
        for ch in n.children:
            cid = build(ch); dot.edge(nid, cid)
        return nid
    build(node); return dot


In [ ]:

# Interfaz
expr_in = widgets.Text(value="total = a + b * ( c + 1 )", description="Entrada:", layout=widgets.Layout(width="100%"))
symtab_in = widgets.Text(value="total:float, a:int, b:int, c:int", description="Símbolos:", layout=widgets.Layout(width="100%"))
run_btn = widgets.Button(description="Analizar semántica", button_style="primary")
out = widgets.Output()

from IPython.display import display, HTML

def parse_symtab(text):
    st = {}; text = text.strip()
    if not text: return st
    for p in [p.strip() for p in text.split(",")]:
        if not p: continue
        if ":" not in p: raise ValueError(f"Entrada inválida: '{p}'. Usa nombre:tipo")
        name, typ = [x.strip() for x in p.split(":",1)]
        if typ not in ("int","float"): raise ValueError(f"Tipo inválido para '{name}': '{typ}'. Usa int|float.")
        st[name] = typ
    return st

def on_run(_):
    out.clear_output()
    with out:
        try:
            tokens, ids = lex(expr_in.value)
            display(HTML("<h4>Tokens</h4>")); display(pd.DataFrame(tokens, columns=["Tipo","Lexema","ini","fin"]))
            display(HTML("<b>Identificadores:</b> " + (", ".join(ids) if ids else "(ninguno)")))
            parser = Parser(tokens); ast = parser.parse()
            symtab = parse_symtab(symtab_in.value)
            errors = []; annotate_types(ast, symtab, errors)
            display(HTML("<h4>AST con tipos</h4>")); display(render_ast(ast))
            if errors:
                display(HTML("<h4 style='color:#b71c1c'>Errores semánticos</h4>")); display(pd.DataFrame({"Error": errors}))
            else:
                display(HTML("<div style='padding:8px;border-left:4px solid #4CAF50'><b>Semántica OK:</b> Tipos consistentes.</div>"))
        except Exception as e:
            display(HTML(f"<div style='color:#b71c1c'><b>Error:</b> {e}</div>"))

run_btn.on_click(on_run)
display(expr_in, symtab_in, run_btn, out)


Text(value='total = a + b * ( c + 1 )', description='Entrada:', layout=Layout(width='100%'))

Text(value='total:float, a:int, b:int, c:int', description='Símbolos:', layout=Layout(width='100%'))

Button(button_style='primary', description='Analizar semántica', style=ButtonStyle())

Output()